# Hierarchical GraphRAG System
**Enterprise · Modular · Deterministic · Local-First**

```
Raw Text → Preprocess → [Modular | LLM Joint] Extraction → Canonical Linking
→ Temporal Enforcement → Graph Build (append-only) → Community Detection (Leiden)
→ Cluster Summarization → Dual Retrieval (Local + Global) → RAG Triad Evaluation
```


In [22]:
# Core (always required)
!pip install spacy rapidfuzz networkx python-dateutil requests

# NER (choose one or all)
!python -m spacy download en_core_web_sm
!pip install transformers gliner

# Community detection (Leiden — optional, falls back to networkx)
!pip install leidenalg python-igraph

# Evaluation
!pip install ragas  # optional full framework

print("Dependency cell — uncomment and run once.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.8 MB/s eta 0:00:0000:0100:01m
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
Dependency cell — uncomment and run once.


## Global Contract (Schema)

Every entity must have a `canonical_id`.  
Every relation must carry `provenance` (source_chunk_id + raw_sentence) and a `time` dict.  
Community nodes are derived **only** from graph edges — no external knowledge injection.


In [39]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional, List, Tuple, Dict, Any
import json, uuid, hashlib, re, os, time, requests


@dataclass
class Entity:
    text: str
    type: str
    span: Tuple[int, int]


@dataclass
class Link:
    entity: str
    canonical_id: str
    confidence: float
    valid_time: Dict[str, Optional[str]]


@dataclass
class Provenance:
    source_chunk_id: str
    raw_sentence: str


@dataclass
class Relation:
    subj: str
    rel: str
    obj: str
    time: Dict[str, Optional[str]]
    provenance: Provenance
    confidence: float


@dataclass
class Chunk:
    doc_id: str
    chunk_id: str
    text: str
    entities: List[Entity] = field(default_factory=list)
    links: List[Link] = field(default_factory=list)
    relations: List[Relation] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "doc_id": self.doc_id, "chunk_id": self.chunk_id, "text": self.text,
            "entities": [{"text": e.text, "type": e.type, "span": list(e.span)} for e in self.entities],
            "links": [{"entity": l.entity, "canonical_id": l.canonical_id,
                       "confidence": l.confidence, "valid_time": l.valid_time} for l in self.links],
            "relations": [{"subj": r.subj, "rel": r.rel, "obj": r.obj, "time": r.time,
                           "provenance": {"source_chunk_id": r.provenance.source_chunk_id,
                                          "raw_sentence": r.provenance.raw_sentence},
                           "confidence": r.confidence} for r in self.relations],
        }


@dataclass
class CommunityNode:
    """Represents one level of the community hierarchy.
    level 0 = leaf cluster, 1 = super-cluster, 2 = global.
    summary is LLM-generated, grounded only in graph edges.
    """
    cluster_id: str
    level: int
    summary: str
    entities: List[str]       # canonical_ids
    key_relations: List[dict] # [{subj, rel, obj, time}]


print("Schema loaded (Entity, Link, Provenance, Relation, Chunk, CommunityNode).")


Schema loaded (Entity, Link, Provenance, Relation, Chunk, CommunityNode).


## 1. Preprocessing Layer

In [40]:
TICKER_PATTERN = re.compile(r'\(([A-Z]{1,5}(?::[A-Z]{2})?)\)|\$([A-Z]{1,5})')


class Preprocessor:
    """Normalize text, extract ticker anchors, chunk into fixed-size windows."""

    def __init__(self, chunk_size: int = 512, ticker_linking: bool = True):
        self.chunk_size = chunk_size
        self.ticker_linking = ticker_linking
        self._ticker_kv: Dict[str, str] = {}

    def extract_tickers(self, text: str) -> List[Tuple[str, int, int]]:
        return [(m.group(1) or m.group(2), m.start(), m.end())
                for m in TICKER_PATTERN.finditer(text)]

    def normalize(self, text: str) -> str:
        return re.sub(r'\s+', ' ', text).strip()

    def chunk(self, doc_id: str, text: str) -> List[Chunk]:
        text = self.normalize(text)
        words = text.split()
        chunks = []
        for i in range(0, max(len(words), 1), self.chunk_size):
            chunk_text = ' '.join(words[i:i + self.chunk_size])
            chunk_id = hashlib.md5(f'{doc_id}:{i}'.encode()).hexdigest()[:12]
            chunks.append(Chunk(doc_id=doc_id, chunk_id=chunk_id, text=chunk_text))
        return chunks

    def anchor_tickers(self, chunk: Chunk) -> Dict[str, str]:
        if not self.ticker_linking:
            return {}
        anchors = {}
        for ticker, _, _ in self.extract_tickers(chunk.text):
            self._ticker_kv.setdefault(ticker, f'TICKER:{ticker}')
            anchors[ticker] = self._ticker_kv[ticker]
        return anchors

    def process(self, doc_id: str, text: str) -> Tuple[List[Chunk], Dict[str, str]]:
        chunks = self.chunk(doc_id, text)
        all_anchors: Dict[str, str] = {}
        for chunk in chunks:
            all_anchors.update(self.anchor_tickers(chunk))
        return chunks, all_anchors


_pp = Preprocessor(chunk_size=50)
_chunks, _anchors = _pp.process('test', 'Goldman Sachs (GS:US) acquired Marcus in 2022.')
print(f'chunks={len(_chunks)}, anchors={_anchors}')


chunks=1, anchors={'GS:US': 'TICKER:GS:US'}


## 2. NER Module

In [41]:
class NERBase(ABC):
    @abstractmethod
    def extract(self, text: str) -> List[Entity]: ...


class SpacyNER(NERBase):
    _model = None
    def __init__(self, model_name: str = 'en_core_web_trf'):
        if SpacyNER._model is None:
            import spacy
            try: SpacyNER._model = spacy.load(model_name)
            except OSError: SpacyNER._model = spacy.load('en_core_web_sm')
    def extract(self, text: str) -> List[Entity]:
        doc = SpacyNER._model(text)
        return [Entity(text=ent.text, type=ent.label_, span=(ent.start_char, ent.end_char))
                for ent in doc.ents]


class FinBERT_NER(NERBase):
    _pipe = None
    MODEL = 'Jean-Baptiste/roberta-large-ner-english'
    def __init__(self):
        if FinBERT_NER._pipe is None:
            from transformers import pipeline as hf_pipeline
            FinBERT_NER._pipe = hf_pipeline('ner', model=self.MODEL, aggregation_strategy='simple')
    def extract(self, text: str) -> List[Entity]:
        return [Entity(text=r['word'], type=r['entity_group'], span=(r['start'], r['end']))
                for r in FinBERT_NER._pipe(text[:512])]


class GLiNER_NER(NERBase):
    _model = None
    LABELS = ['organization', 'person', 'location', 'product', 'financial instrument', 'event']
    def __init__(self):
        if GLiNER_NER._model is None:
            from gliner import GLiNER
            GLiNER_NER._model = GLiNER.from_pretrained('urchade/gliner_medium-v2.1')
    def extract(self, text: str) -> List[Entity]:
        return [Entity(text=e['text'], type=e['label'].upper(), span=(e['start'], e['end']))
                for e in GLiNER_NER._model.predict_entities(text, self.LABELS)]


NER_REGISTRY: Dict[str, type] = {
    'spacy_trf': SpacyNER,
    'finbert_ner': FinBERT_NER,
    'gliner': GLiNER_NER,
}

def get_ner(name: str) -> NERBase:
    if name not in NER_REGISTRY:
        raise ValueError(f'Unknown NER: {name}. Available: {list(NER_REGISTRY)}')
    return NER_REGISTRY[name]()

print('NER registry:', list(NER_REGISTRY))


NER registry: ['spacy_trf', 'finbert_ner', 'gliner']


## 3. NEL Module (Hierarchical Linking)

In [42]:
from rapidfuzz import fuzz, process as rf_process

ALIAS_TABLE: Dict[str, str] = {
    'Goldman Sachs': 'LEI:549300IUPX6KIT9GK103', 'Goldman': 'LEI:549300IUPX6KIT9GK103',
    'GS': 'LEI:549300IUPX6KIT9GK103', 'Apple': 'GID:AAPL', 'Apple Inc': 'GID:AAPL',
    'Microsoft': 'GID:MSFT', 'JP Morgan': 'LEI:7H6GLXDRUGQFU57RNE97',
    'JPMorgan': 'LEI:7H6GLXDRUGQFU57RNE97', 'JP Morgan Chase': 'LEI:7H6GLXDRUGQFU57RNE97',
    'OpenAI': 'GID:OPENAI', 'First Republic Bank': 'GID:FRB',
    'Marcus': 'LEI:549300IUPX6KIT9GK103_MARCUS',
}

CANONICAL_DB: Dict[str, dict] = {
    'LEI:549300IUPX6KIT9GK103': {'canonical_id': 'LEI:549300IUPX6KIT9GK103',
        'name': 'Goldman Sachs Group Inc', 'type': 'parent', 'parent_id': None,
        'valid_time': {'start': '1869-01-01', 'end': None}},
    'LEI:549300IUPX6KIT9GK103_MARCUS': {'canonical_id': 'LEI:549300IUPX6KIT9GK103_MARCUS',
        'name': 'Marcus by Goldman Sachs', 'type': 'subsidiary',
        'parent_id': 'LEI:549300IUPX6KIT9GK103', 'valid_time': {'start': '2016-10-01', 'end': None}},
    'GID:AAPL': {'canonical_id': 'GID:AAPL', 'name': 'Apple Inc', 'type': 'parent',
        'parent_id': None, 'valid_time': {'start': '1977-01-01', 'end': None}},
    'GID:MSFT': {'canonical_id': 'GID:MSFT', 'name': 'Microsoft Corporation', 'type': 'parent',
        'parent_id': None, 'valid_time': {'start': '1975-04-04', 'end': None}},
    'LEI:7H6GLXDRUGQFU57RNE97': {'canonical_id': 'LEI:7H6GLXDRUGQFU57RNE97',
        'name': 'JPMorgan Chase & Co', 'type': 'parent', 'parent_id': None,
        'valid_time': {'start': '1799-01-01', 'end': None}},
}


class NEL:
    """Named Entity Linking: Entity text -> canonical_id (LEI / GID / LOCAL).
    Stages: ticker → alias → fuzzy → embedding stub → local fingerprint.
    """
    def __init__(self, ticker_anchors: Dict[str, str], strategy: str = 'fuzzy_flat',
                 fuzzy_threshold: float = 75.0):
        self.ticker_anchors = dict(ticker_anchors)
        self.strategy = strategy
        self.fuzzy_threshold = fuzzy_threshold
        self._alias_keys = list(ALIAS_TABLE)

    def link(self, entity: Entity, context: str = '') -> Link:
        canonical_id: Optional[str] = None
        confidence = 0.0

        cid = self.ticker_anchors.get(entity.text.upper())
        if cid: canonical_id, confidence = cid, 1.0

        if not canonical_id:
            cid = ALIAS_TABLE.get(entity.text)
            if cid: canonical_id, confidence = cid, 0.95

        if not canonical_id:
            match, score, _ = rf_process.extractOne(entity.text, self._alias_keys, scorer=fuzz.token_sort_ratio)
            if score >= self.fuzzy_threshold:
                canonical_id, confidence = ALIAS_TABLE[match], score / 100.0

        if not canonical_id:
            canonical_id = f"LOCAL:{hashlib.md5(entity.text.lower().encode()).hexdigest()[:8]}"
            confidence = 0.0

        valid_time = CANONICAL_DB.get(canonical_id, {}).get('valid_time', {'start': None, 'end': None})
        return Link(entity=entity.text, canonical_id=canonical_id,
                    confidence=confidence, valid_time=valid_time)

    def link_all(self, entities: List[Entity], context: str = '') -> List[Link]:
        return [self.link(e, context) for e in entities]


_nel_test = NEL(ticker_anchors={'GS': 'LEI:549300IUPX6KIT9GK103'})
_lnk = _nel_test.link(Entity('Goldman Sachs', 'ORG', (0, 13)))
print(f'Link: {_lnk.entity} -> {_lnk.canonical_id} ({_lnk.confidence:.2f})')


Link: Goldman Sachs -> LEI:549300IUPX6KIT9GK103 (0.95)


## 4. Temporal Layer (Point-in-Time Enforcement)

In [43]:
from dateutil import parser as dateparser

_DATE_PATS = [
    re.compile(r'\b(\d{4}-\d{2}-\d{2})\b'),
    re.compile(r'\b(\w+ \d{1,2},?\s+\d{4})\b'),
    re.compile(r'\b(Q[1-4]\s+\d{4})\b', re.IGNORECASE),
    re.compile(r'\b(early|mid|late)\s+(\d{4})\b', re.IGNORECASE),
    re.compile(r'\b(\d{4})\b'),
]


class TemporalLayer:
    def extract_dates(self, text: str) -> List[Tuple[str, int, int]]:
        found = []
        for pat in _DATE_PATS:
            for m in pat.finditer(text):
                found.append((m.group(), m.start(), m.end()))
        found.sort(key=lambda x: x[1])
        deduped, last_end = [], -1
        for item in found:
            if item[1] >= last_end:
                deduped.append(item)
                last_end = item[2]
        return deduped

    def normalize_date(self, date_str: str) -> Optional[str]:
        s = date_str.strip()
        if re.fullmatch(r'\d{4}-\d{2}-\d{2}', s):
            return s

        # Quarter format: Q1 2022 -> 2022-01-01
        q = re.fullmatch(r'Q([1-4])\s+(\d{4})', s, re.IGNORECASE)
        if q:
            year = q.group(2)
            month = (int(q.group(1)) - 1) * 3 + 1
            return f'{year}-{month:02d}-01'

        # Coarse period: early/mid/late 2023
        p = re.fullmatch(r'(early|mid|late)\s+(\d{4})', s, re.IGNORECASE)
        if p:
            month_map = {'early': '01', 'mid': '06', 'late': '10'}
            return f"{p.group(2)}-{month_map[p.group(1).lower()]}-01"

        # Year-only: 2024 -> 2024-01-01
        if re.fullmatch(r'\d{4}', s):
            return f'{s}-01-01'

        # Fallback parser 
        try:
            dt = dateparser.parse(s, fuzzy=True)
            return dt.strftime('%Y-%m-%d') if dt else None
        except Exception:
            return None

    def infer_relation_time(self, sentence: str, doc_date: Optional[str] = None) -> Dict[str, Optional[str]]:
        normalized = sorted(filter(None, (self.normalize_date(d[0]) for d in self.extract_dates(sentence))))
        if normalized:
            return {'start': normalized[0], 'end': normalized[-1] if len(normalized) > 1 else None}
        return {'start': doc_date, 'end': None}

    def is_valid(self, relation_time: Dict, entity_valid_time: Dict) -> bool:
        r_start = relation_time.get('start')
        e_start, e_end = entity_valid_time.get('start'), entity_valid_time.get('end')
        if not r_start: return True
        if e_end and r_start > e_end: return False
        if e_start and r_start < e_start: return False
        return True


_tl = TemporalLayer()
print(_tl.normalize_date('Q1 2022'), _tl.normalize_date('early 2023'))

2022-01-01 2023-01-01


## 5. RE Module (Relation Extraction)

Two paradigms — **modular** (Snowball | OpenIE) vs **joint LLM** (Ollama / Llama 3.2).  
`Ollama_RE` is the primary LLM extractor: strict JSON schema, temp=0, local inference.


In [44]:
class REBase(ABC):
    @abstractmethod
    def extract(self, text: str, entities: List[Entity]) -> List[Relation]: ...


# ── A. OpenIE (SVO pattern fallback) ─────────────────────────────────────────
class OpenIE_RE(REBase):
    _SVO = re.compile(
        r'(?P<subj>[A-Z][\w ]{2,30}?)\s+(?P<verb>\w+(?:ed|s|ing)?)\s+(?P<obj>[A-Z][\w ]{2,30})'
    )
    def __init__(self):
        try: import openie; self._openie = openie
        except ImportError: self._openie = None

    def extract(self, text: str, entities: List[Entity]) -> List[Relation]:
        entity_texts = {e.text for e in entities}
        rels = []
        for m in self._SVO.finditer(text):
            subj, verb, obj = m.group('subj').strip(), m.group('verb'), m.group('obj').strip()
            if any(e.lower() in subj.lower() or e.lower() in obj.lower() for e in entity_texts):
                rels.append(Relation(subj=subj, rel=verb, obj=obj,
                    time={'start': None, 'end': None},
                    provenance=Provenance('', text[:300]), confidence=0.4))
        return rels


# ── B. Snowball (seeded financial patterns) ───────────────────────────────────
class Snowball_RE(REBase):
    _SEEDS = [
        (re.compile(r'(?P<subj>[A-Z][\w ]{1,30}?)(?:\s*\([^)]*\))?\s+acquired\s+(?P<obj>[A-Z][\w ]{1,30})', re.I), 'acquired'),
        (re.compile(r'(?P<subj>[A-Z][\w ]{1,30}?)(?:\s*\([^)]*\))?\s+merged\s+with\s+(?P<obj>[A-Z][\w ]{1,30})', re.I), 'merged_with'),
        (re.compile(r'(?P<subj>[A-Z][\w ]{1,30}?)(?:\s*\([^)]*\))?\s+(?:invested|investment)\s+in\s+(?P<obj>[A-Z][\w ]{1,30})', re.I), 'invested_in'),
        (re.compile(r'(?P<subj>[A-Z][\w ]{1,30}?)(?:\s*\([^)]*\))?\s+partnered\s+with\s+(?P<obj>[A-Z][\w ]{1,30})', re.I), 'partner_of'),
        (re.compile(r'(?P<subj>[A-Z][\w ]{1,30}?)(?:\s*\([^)]*\))?\s+sued\s+(?P<obj>[A-Z][\w ]{1,30})', re.I), 'sued'),
        (re.compile(r'(?P<subj>[A-Z][\w ]{1,30}?)(?:\s*\([^)]*\))?\s+sold\s+(?:its\s+)?(?:stake|shares|unit|division)\s+to\s+(?P<obj>[A-Z][\w ]{1,30})', re.I), 'sold_stake_to'),
        (re.compile(r'(?P<subj>[A-Z][\w ]{1,30}?)(?:\s*\([^)]*\))?\s+raised\s+\$[\d.]+[BMK]?\s+(?:from\s+)?(?P<obj>[A-Z][\w ]{1,30})', re.I), 'raised_funding_from'),
    ]

    def extract(self, text: str, entities: List[Entity]) -> List[Relation]:
        entity_texts = {e.text for e in entities}
        def _best(span):
            for et in entity_texts:
                if et.lower() in span.lower(): return et
            return span.strip()
        rels = []
        for pattern, rel_type in self._SEEDS:
            for m in pattern.finditer(text):
                subj, obj = _best(m.group('subj')), _best(m.group('obj'))
                if subj.lower() == obj.lower(): continue
                if subj not in entity_texts and obj not in entity_texts: continue
                rels.append(Relation(subj=subj, rel=rel_type, obj=obj,
                    time={'start': None, 'end': None},
                    provenance=Provenance('', text[:300]), confidence=0.7))
        return rels


# ── C. Ollama_RE (Llama 3.2 — joint LLM extraction) ──────────────────────────
class Ollama_RE(REBase):
    """Llama 3.2 via Ollama REST API. temperature=0, strict JSON contract.
    dev_mode=True → llama3.2:1b for fast iteration.
    """

    RELATION_TYPES = [
        'acquired', 'merged_with', 'invested_in', 'partner_of', 'subsidiary_of',
        'sued', 'sold_stake_to', 'raised_funding_from', 'ceo_of', 'competes_with',
        'regulates', 'board_member_of',
    ]

    _PROMPT = """Extract ONLY valid financial relations as a JSON array.
Each element: {{"subj": str, "rel": str, "obj": str, "time": str|null, "provenance": str}}

Rules:
- resolve coreference before extraction
- no hallucinated entities
- provenance = exact source sentence
- rel MUST be one of: {rel_types}
- return [] if no valid relations found

Text: {text}
Known entities: {entities}

Output ONLY the JSON array, nothing else:"""

    def __init__(self, model: str = 'llama3.2:3b', dev_mode: bool = False,
                 ollama_url: str = 'http://localhost:11434'):
        self.model = 'llama3.2:1b' if dev_mode else model
        self.ollama_url = ollama_url

    def _call(self, prompt: str) -> str:
        resp = requests.post(f'{self.ollama_url}/api/generate',
            json={'model': self.model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0, 'top_p': 1}},
            timeout=120)
        resp.raise_for_status()
        return resp.json()['response']

    def extract(self, text: str, entities: List[Entity]) -> List[Relation]:
        prompt = self._PROMPT.format(
            rel_types=', '.join(self.RELATION_TYPES),
            text=text[:1500],
            entities=json.dumps([{'text': e.text, 'type': e.type} for e in entities])
        )
        try:
            raw = self._call(prompt)
            arr_match = re.search(r'\[.*\]', raw, re.DOTALL)
            if not arr_match:
                return []
            triples = json.loads(arr_match.group())
            rels = []
            for t in triples:
                if not all(k in t for k in ('subj', 'rel', 'obj')): continue
                if t['rel'] not in self.RELATION_TYPES: continue
                t_time = t.get('time')
                rels.append(Relation(
                    subj=t['subj'], rel=t['rel'], obj=t['obj'],
                    time={'start': t_time, 'end': None} if isinstance(t_time, str)
                         else (t_time or {'start': None, 'end': None}),
                    provenance=Provenance('', t.get('provenance', text[:300])),
                    confidence=float(t.get('confidence', 0.8))
                ))
            return rels
        except Exception as exc:
            print(f'[Ollama_RE] {exc}')
            return []


RE_REGISTRY: Dict[str, type] = {
    'openie':   OpenIE_RE,
    'snowball': Snowball_RE,
    'ollama':   Ollama_RE,
}

def get_re(name: str, **kwargs) -> REBase:
    if name not in RE_REGISTRY:
        raise ValueError(f'Unknown RE: {name}. Available: {list(RE_REGISTRY)}')
    return RE_REGISTRY[name](**kwargs)

print('RE registry:', list(RE_REGISTRY))


RE registry: ['openie', 'snowball', 'ollama']


## 6. Provenance Enforcement

In [45]:
class ProvenanceEnforcer:
    @staticmethod
    def enforce(r: Relation, chunk_id: str, fallback: str = '') -> Relation:
        if not r.provenance.source_chunk_id: r.provenance.source_chunk_id = chunk_id
        if not r.provenance.raw_sentence: r.provenance.raw_sentence = fallback
        return r

    @staticmethod
    def validate(r: Relation) -> Tuple[bool, str]:
        if not r.provenance.source_chunk_id: return False, 'missing source_chunk_id'
        if not r.provenance.raw_sentence: return False, 'missing raw_sentence'
        return True, 'ok'

    @staticmethod
    def filter_valid(relations: List[Relation]) -> List[Relation]:
        out = []
        for r in relations:
            ok, reason = ProvenanceEnforcer.validate(r)
            if ok: out.append(r)
            else: print(f'[Provenance] Dropped ({r.subj},{r.rel},{r.obj}): {reason}')
        return out

print('ProvenanceEnforcer loaded.')


ProvenanceEnforcer loaded.


## 7. Graph Builder (Versioned, Append-Only)

In [46]:
import networkx as nx
from collections import defaultdict


class GraphBuilder:
    """Append-only versioned knowledge graph.
    backend: 'networkx' (dev) | 'neo4j' (prod stub)
    scope:   'global' | 'document_local'
    Edge key: (subj_id, obj_id, rel, time_start) — parallel edges, never overwritten.
    """

    def __init__(self, backend: str = 'networkx', scope: str = 'global'):
        self.backend = backend
        self.scope = scope
        self._local: Dict[str, nx.MultiDiGraph] = {}
        self._global: nx.MultiDiGraph = nx.MultiDiGraph()
        self._edge_index: Dict[Tuple, List[dict]] = defaultdict(list)

    def _graph(self, doc_id: str) -> nx.MultiDiGraph:
        if self.scope == 'document_local':
            if doc_id not in self._local:
                self._local[doc_id] = nx.MultiDiGraph()
            return self._local[doc_id]
        return self._global

    def _add_node(self, G, cid, **attrs):
        if not G.has_node(cid): G.add_node(cid, **attrs)
        else: G.nodes[cid].update({k: v for k, v in attrs.items() if v is not None})

    def add_relation(self, doc_id: str, ls: Link, lo: Link, r: Relation) -> None:
        G = self._graph(doc_id)
        self._add_node(G, ls.canonical_id, name=ls.entity, valid_time=ls.valid_time, confidence=ls.confidence)
        self._add_node(G, lo.canonical_id, name=lo.entity, valid_time=lo.valid_time, confidence=lo.confidence)
        time_start = r.time.get('start') or 'unknown'
        idx_key = (ls.canonical_id, lo.canonical_id, r.rel, time_start)
        edge_data = {'rel': r.rel, 'time': r.time,
            'provenance': {'source_chunk_id': r.provenance.source_chunk_id,
                           'raw_sentence': r.provenance.raw_sentence},
            'confidence': r.confidence, 'doc_id': doc_id}
        edge_key = f"{r.rel}__{time_start}__{len(self._edge_index[idx_key])}"
        G.add_edge(ls.canonical_id, lo.canonical_id, key=edge_key, **edge_data)
        self._edge_index[idx_key].append(edge_data)

    def get_graph(self, doc_id: Optional[str] = None) -> nx.MultiDiGraph:
        if self.scope == 'document_local':
            if doc_id: return self._local.get(doc_id, nx.MultiDiGraph())
            merged = nx.MultiDiGraph()
            for g in self._local.values(): merged.update(g)
            return merged
        return self._global

    def stats(self) -> dict:
        G = self.get_graph()
        total = G.number_of_edges()
        with_prov = sum(1 for _,_,d in G.edges(data=True) if d.get('provenance',{}).get('source_chunk_id'))
        with_ts = sum(1 for _,_,d in G.edges(data=True) if d.get('time',{}).get('start'))
        return {'nodes': G.number_of_nodes(), 'edges': total,
                'provenance_pct': round(with_prov/max(total,1)*100,1),
                'timestamp_pct': round(with_ts/max(total,1)*100,1),
                'conflict_groups': len({k:v for k,v in self._edge_index.items() if len(v)>1})}

print('GraphBuilder loaded.')


GraphBuilder loaded.


## 8. Community Detection (Leiden)

Leiden algorithm applied to the undirected projection of the knowledge graph.  
Falls back to `networkx.community.greedy_modularity_communities` if `leidenalg` is not installed.  
Produces a **3-level hierarchy**: leaf clusters (L0) → super-clusters (L1) → global (L2).


In [47]:
class CommunityDetector:
    """Detect communities with Leiden; build 3-level hierarchy of CommunityNode objects."""

    def __init__(self, resolution: float = 1.0, n_iterations: int = 10):
        self.resolution = resolution
        self.n_iterations = n_iterations

    # ── low-level: run Leiden (or networkx fallback) ──────────────────────────
    def _partition(self, G_und: nx.Graph) -> Dict[str, int]:
        nodes = list(G_und.nodes())
        try:
            import leidenalg, igraph as ig
            node_idx = {n: i for i, n in enumerate(nodes)}
            edges = [(node_idx[u], node_idx[v]) for u, v in G_und.edges()]
            ig_g = ig.Graph(n=len(nodes), edges=edges, directed=False)
            part = leidenalg.find_partition(
                ig_g, leidenalg.ModularityVertexPartition,
                n_iterations=self.n_iterations
            )
            return {nodes[i]: c for c, members in enumerate(part) for i in members}
        except ImportError:
            comms = list(nx.community.greedy_modularity_communities(G_und))
            return {node: i for i, comm in enumerate(comms) for node in comm}

    # ── public: map every node to its leaf cluster id ─────────────────────────
    def detect(self, G: nx.MultiDiGraph) -> Dict[str, int]:
        """Return {canonical_id -> cluster_id} for every node in G."""
        if G.number_of_nodes() == 0:
            return {}
        G_und = G.to_undirected()
        connected = [n for n in G_und.nodes() if G_und.degree(n) > 0]
        if not connected:
            return {n: 0 for n in G_und.nodes()}
        mapping = self._partition(G_und.subgraph(connected))
        # isolated nodes get cluster -1
        for n in G_und.nodes():
            mapping.setdefault(n, -1)
        return mapping

    # ── build 3-level hierarchy ───────────────────────────────────────────────
    def build_hierarchy(
        self,
        G: nx.MultiDiGraph,
        level0: Dict[str, int],
    ) -> Dict[int, CommunityNode]:
        communities: Dict[int, CommunityNode] = {}
        cluster_members: Dict[int, List[str]] = defaultdict(list)
        for node, cid in level0.items():
            cluster_members[cid].append(node)

        # Level 0 — leaf clusters
        for cid, members in cluster_members.items():
            key_rels = [
                {'subj': u, 'rel': d.get('rel',''), 'obj': v, 'time': d.get('time',{})}
                for u, v, d in G.edges(data=True)
                if u in members and v in members
            ][:20]
            communities[cid] = CommunityNode(
                cluster_id=f'L0_C{cid}', level=0, summary='',
                entities=list(members), key_relations=key_rels
            )

        # Level 1 — super-clusters: connected components of cluster adjacency
        cluster_adj: Dict[int, set] = defaultdict(set)
        for u, v in G.edges():
            cu, cv = level0.get(u, -1), level0.get(v, -1)
            if cu != cv and cu != -1 and cv != -1:
                cluster_adj[cu].add(cv)
                cluster_adj[cv].add(cu)

        visited: set = set()
        sc_id = 0
        node_to_sc: Dict[int, int] = {}
        for cid in cluster_members:
            if cid in visited: continue
            queue, group = [cid], []
            while queue:
                curr = queue.pop()
                if curr in visited: continue
                visited.add(curr); group.append(curr)
                queue.extend(cluster_adj[curr] - visited)
            for g in group:
                node_to_sc[g] = sc_id
            sc_id += 1

        for sid in range(sc_id):
            children = [c for c, s in node_to_sc.items() if s == sid]
            all_ents = [e for c in children for e in communities[c].entities]
            all_rels = [r for c in children for r in communities[c].key_relations][:30]
            new_id = max(communities, default=-1) + 1
            communities[new_id] = CommunityNode(
                cluster_id=f'L1_SC{sid}', level=1, summary='',
                entities=all_ents, key_relations=all_rels
            )

        print(f'Hierarchy built: {sum(1 for n in communities.values() if n.level==0)} L0 clusters, '
              f'{sum(1 for n in communities.values() if n.level==1)} L1 super-clusters.')
        return communities


print('CommunityDetector loaded.')


CommunityDetector loaded.


## 9. Community Summaries (LLM-Generated)

Each cluster gets an LLM-generated summary via Ollama.  
**Constraint**: summaries are grounded *only* in graph edges — no external knowledge.


In [48]:
class CommunitySummarizer:
    """Generate CommunityNode summaries via Ollama. Grounded in graph edges only."""

    _PROMPT = """You are a financial analyst. Summarize this entity cluster using ONLY the relations listed.
Do NOT add external knowledge. Do NOT invent facts not present in the relations.

Cluster ID: {cluster_id}
Level: {level}
Entities: {entities}
Relations:
{relations}

Respond with ONLY this JSON:
{{
  "cluster_id": "{cluster_id}",
  "level": {level},
  "summary": "<2-3 sentences grounded strictly in the relations above>",
  "entities": {entities_json},
  "key_relations": {rels_json}
}}"""

    def __init__(self, model: str = 'llama3.2:3b', dev_mode: bool = False,
                 ollama_url: str = 'http://localhost:11434'):
        self.model = 'llama3.2:1b' if dev_mode else model
        self.ollama_url = ollama_url

    def _call(self, prompt: str) -> str:
        resp = requests.post(f'{self.ollama_url}/api/generate',
            json={'model': self.model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0}},
            timeout=120)
        resp.raise_for_status()
        return resp.json()['response']

    def summarize(self, node: CommunityNode) -> CommunityNode:
        rels_str = '\n'.join(
            f'  ({r["subj"]}) --[{r["rel"]}]--> ({r["obj"]})'
            + (f' @ {r["time"].get("start","?")}' if r.get('time') else '')
            for r in node.key_relations[:15]
        ) or '  (no intra-cluster relations)'

        prompt = self._PROMPT.format(
            cluster_id=node.cluster_id, level=node.level,
            entities=node.entities[:20], relations=rels_str,
            entities_json=json.dumps(node.entities[:20]),
            rels_json=json.dumps(node.key_relations[:10])
        )
        try:
            raw = self._call(prompt)
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            if m:
                data = json.loads(m.group())
                node.summary = data.get('summary', '')
        except Exception as exc:
            print(f'[CommunitySummarizer] {node.cluster_id}: {exc}')
            node.summary = f'[Summarization failed for {node.cluster_id}]'
        return node

    def summarize_all(self, communities: Dict[int, CommunityNode]) -> Dict[int, CommunityNode]:
        print(f'Summarizing {len(communities)} community nodes...')
        for cid in sorted(communities, key=lambda k: communities[k].level):
            node = communities[cid]
            print(f'  {node.cluster_id} (L{node.level}, {len(node.entities)} entities)')
            communities[cid] = self.summarize(node)
        return communities


print('CommunitySummarizer loaded.')


CommunitySummarizer loaded.


## 10. Dual Retrieval System

| Mode | Trigger | Mechanism |
|------|---------|-----------|
| **Local** (KG-RAG) | Query contains named entities | Graph traversal from seed nodes |
| **Global** (Hierarchical) | Thematic / no entities | Keyword-score cluster summaries |
| **Hybrid** | Explicit | Local context + Global context merged |

Routing is automatic (`mode='auto'`): entity presence → local, else → global.


In [49]:
class LocalRetriever:
    """KG-RAG: entity extraction → subgraph traversal → natural-language context."""

    def __init__(self, graph: nx.MultiDiGraph, nel: NEL,
                 max_hops: int = 2, max_nodes: int = 25):
        self.graph = graph
        self.nel = nel
        self.max_hops = max_hops
        self.max_nodes = max_nodes

    def _query_entities(self, query: str) -> List[str]:
        """Map query mentions to canonical_ids via ticker + alias lookup."""
        found = []
        for m in TICKER_PATTERN.finditer(query):
            found.append(f'TICKER:{m.group(1) or m.group(2)}')
        for alias, cid in ALIAS_TABLE.items():
            if alias.lower() in query.lower():
                found.append(cid)
        return list(set(found))

    def _traverse(self, seeds: List[str]) -> nx.MultiDiGraph:
        seeds = [s for s in seeds if s in self.graph]
        if not seeds:
            return nx.MultiDiGraph()
        visited = set(seeds)
        frontier = list(seeds)
        for _ in range(self.max_hops):
            nxt = []
            for node in frontier:
                for nbr in list(self.graph.successors(node)) + list(self.graph.predecessors(node)):
                    if nbr not in visited:
                        visited.add(nbr); nxt.append(nbr)
            frontier = nxt
            if len(visited) >= self.max_nodes: break
        return self.graph.subgraph(list(visited)[:self.max_nodes]).copy()

    def _to_context(self, sg: nx.MultiDiGraph) -> str:
        if sg.number_of_edges() == 0:
            return 'No graph context found for the query entities.'
        lines = ['[Local KG Context]']
        for u, v, d in sg.edges(data=True):
            un = sg.nodes[u].get('name', u)
            vn = sg.nodes[v].get('name', v)
            ts = d.get('time', {}).get('start', '?')
            prov = d.get('provenance', {}).get('raw_sentence', '')[:100]
            lines.append(f'  ({un}) --[{d.get("rel","?")}]--> ({vn}) @ {ts}')
            if prov: lines.append(f'    \""{prov}\"')
        return '\n'.join(lines)

    def retrieve(self, query: str) -> dict:
        eids = self._query_entities(query)
        sg = self._traverse(eids)
        return {'context': self._to_context(sg), 'retrieval_mode': 'local',
                'entity_ids': eids, 'subgraph_nodes': sg.number_of_nodes()}


class GlobalRetriever:
    """Hierarchical RAG: keyword-score cluster summaries → retrieve top-k → context."""

    def __init__(self, communities: Dict[int, CommunityNode], top_k: int = 3):
        self.communities = communities
        self.top_k = top_k

    def _score(self, query: str, node: CommunityNode) -> float:
        qterms = set(query.lower().split())
        s_score = len(qterms & set(node.summary.lower().split())) / (len(qterms) + 1)
        e_terms = {t.lower() for e in node.entities
                   for t in e.replace(':', ' ').replace('_', ' ').split()}
        e_score = len(qterms & e_terms) / (len(qterms) + 1)
        r_terms = {t.lower() for r in node.key_relations
                   for t in (r.get('subj','') + ' ' + r.get('obj','')).split()}
        r_score = len(qterms & r_terms) / (len(qterms) + 1)
        return s_score * 0.5 + e_score * 0.3 + r_score * 0.2

    def retrieve(self, query: str) -> dict:
        if not self.communities:
            return {'context': '[No communities available]', 'retrieval_mode': 'global', 'clusters': []}
        scored = sorted(
            [(cid, n, self._score(query, n)) for cid, n in self.communities.items()],
            key=lambda x: x[2], reverse=True
        )[:self.top_k]
        lines = ['[Global Hierarchical Context]']
        clusters_used = []
        for _, node, score in scored:
            if not node.summary: continue
            lines.append(f'\n[{node.cluster_id}] L{node.level} (score={score:.3f}):')
            lines.append(f'  {node.summary}')
            lines.append(f'  Entities: {", ".join(str(e) for e in node.entities[:8])}')
            clusters_used.append(node.cluster_id)
        return {'context': '\n'.join(lines), 'retrieval_mode': 'global', 'clusters': clusters_used}


class DualRetriever:
    """Routes to Local or Global retrieval; generates answer via Ollama.
    Routing: entity found in query → local; else → global.
    """

    def __init__(self, local: LocalRetriever, global_: GlobalRetriever,
                 model: str = 'llama3.2:3b', ollama_url: str = 'http://localhost:11434'):
        self.local = local
        self.global_ = global_
        self.model = model
        self.ollama_url = ollama_url

    def _has_entities(self, query: str) -> bool:
        if TICKER_PATTERN.search(query): return True
        return any(a.lower() in query.lower() for a in ALIAS_TABLE)

    def retrieve(self, query: str, mode: str = 'auto') -> dict:
        if mode == 'local': return self.local.retrieve(query)
        if mode == 'global': return self.global_.retrieve(query)
        if mode == 'hybrid':
            l, g = self.local.retrieve(query), self.global_.retrieve(query)
            return {'context': l['context'] + '\n\n' + g['context'],
                    'retrieval_mode': 'hybrid', 'local': l, 'global': g}
        # auto
        return self.local.retrieve(query) if self._has_entities(query) else self.global_.retrieve(query)

    def answer(self, query: str, mode: str = 'auto') -> dict:
        ret = self.retrieve(query, mode)
        prompt = (
            'Answer the question using ONLY the provided knowledge graph context.\n'
            'If context is insufficient, say "Insufficient context."\n\n'
            f'Context:\n{ret["context"]}\n\nQuestion: {query}\n\nAnswer:'
        )
        try:
            resp = requests.post(f'{self.ollama_url}/api/generate',
                json={'model': self.model, 'prompt': prompt, 'stream': False,
                      'options': {'temperature': 0}},
                timeout=60)
            resp.raise_for_status()
            answer = resp.json()['response']
        except Exception as exc:
            answer = f'[Answer generation failed: {exc}]'
        return {'query': query, 'answer': answer,
                'context': ret['context'], 'retrieval_mode': ret['retrieval_mode']}


print('LocalRetriever, GlobalRetriever, DualRetriever loaded.')


LocalRetriever, GlobalRetriever, DualRetriever loaded.


## 10b. Advanced Retrieval — LLM-Guided + Semantic

Upgrades over the rule-based `LocalRetriever` / keyword `GlobalRetriever`:

| Class | Upgrade |
|-------|---------|
| **`LLM_LocalRetriever`** | `llama3.2:3b` extracts query entities; guided multi-hop prunes irrelevant neighbours |
| **`SemanticGlobalRetriever`** | `sentence-transformers` cosine similarity replaces keyword overlap; falls back gracefully |
| **`AdvancedDualRetriever`** | New `llm_hybrid` mode: LLM synthesises + re-ranks both local and global contexts |

In [50]:
# ── A. LLM_LocalRetriever ────────────────────────────────────────────────────
class LLM_LocalRetriever(LocalRetriever):
    """Extends LocalRetriever with llama3.2:3b-guided entity extraction.

    Two upgrades over rule-based LocalRetriever:
      1. LLM extracts query entities  (not just ticker regex / alias table)
      2. LLM prunes neighbour expansion (guided multi-hop vs blind BFS)
    """

    _ENTITY_PROMPT = (
        "Identify all named entities (organizations, people, financial instruments, products) "
        "in the following query. Return ONLY a JSON array of strings. No explanation.\n\n"
        "Query: {query}\n\n"
        "Known entity names for context (match if applicable):\n{known_ids}\n\n"
        "Output ONLY JSON array, e.g. [\"Goldman Sachs\", \"Apple\"]:"
    )

    _PRUNE_PROMPT = (
        "Given a knowledge graph traversal for the query below, "
        "select which neighbour nodes are most relevant to expand.\n\n"
        "Query: {query}\n"
        "Current nodes: {current_nodes}\n"
        "Candidate neighbours: {candidates}\n\n"
        "Return ONLY a JSON array of node names to expand (max {max_expand} items).\n"
        "Output ONLY the JSON array:"
    )

    def __init__(self, graph: nx.MultiDiGraph, nel: NEL,
                 max_hops: int = 2, max_nodes: int = 25,
                 model: str = 'llama3.2:3b',
                 ollama_url: str = 'http://localhost:11434'):
        super().__init__(graph, nel, max_hops, max_nodes)
        self.model = model
        self.ollama_url = ollama_url

    def _llm_call(self, prompt: str, timeout: int = 30) -> str:
        resp = requests.post(
            f'{self.ollama_url}/api/generate',
            json={'model': self.model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0, 'top_p': 1}},
            timeout=timeout
        )
        resp.raise_for_status()
        return resp.json()['response']

    def _llm_query_entities(self, query: str) -> List[str]:
        """Use llama3.2:3b to extract entity mentions, then resolve to canonical_ids."""
        known = list(ALIAS_TABLE.keys())[:20]
        prompt = self._ENTITY_PROMPT.format(query=query, known_ids=json.dumps(known))
        try:
            raw = self._llm_call(prompt)
            arr = re.search(r'\[.*?\]', raw, re.DOTALL)
            if not arr:
                return self._query_entities(query)   # rule-based fallback
            mentions = json.loads(arr.group())
            from rapidfuzz import process as rf_proc, fuzz as _fuzz
            resolved = []
            for mention in mentions:
                cid = ALIAS_TABLE.get(mention)
                if cid:
                    resolved.append(cid)
                    continue
                mr = rf_proc.extractOne(mention, list(ALIAS_TABLE.keys()),
                                        scorer=_fuzz.token_sort_ratio)
                if mr and mr[1] >= 70:
                    resolved.append(ALIAS_TABLE[mr[0]])
                else:
                    local_id = f"LOCAL:{hashlib.md5(mention.lower().encode()).hexdigest()[:8]}"
                    if local_id in self.graph:
                        resolved.append(local_id)
            return list(set(resolved + self._query_entities(query)))
        except Exception as exc:
            print(f'[LLM_LocalRetriever] entity extraction failed: {exc}, using rule-based')
            return self._query_entities(query)

    def _llm_guided_traverse(self, seeds: List[str], query: str) -> nx.MultiDiGraph:
        """Multi-hop traversal where llama3.2:3b prunes which neighbours to expand."""
        seeds = [s for s in seeds if s in self.graph]
        if not seeds:
            return nx.MultiDiGraph()

        visited = set(seeds)
        frontier = list(seeds)

        for hop in range(self.max_hops):
            candidates: Dict[str, str] = {}
            for node in frontier:
                for nbr in list(self.graph.successors(node)) + list(self.graph.predecessors(node)):
                    if nbr not in visited:
                        candidates[nbr] = self.graph.nodes[nbr].get('name', nbr)

            if not candidates:
                break

            max_expand = max(1, self.max_nodes - len(visited))
            if len(candidates) <= max_expand:
                next_frontier = list(candidates.keys())
            else:
                current_names = [self.graph.nodes[n].get('name', n) for n in visited]
                prompt = self._PRUNE_PROMPT.format(
                    query=query,
                    current_nodes=json.dumps(current_names[:15]),
                    candidates=json.dumps(list(candidates.values())[:20]),
                    max_expand=max_expand,
                )
                try:
                    raw = self._llm_call(prompt)
                    arr = re.search(r'\[.*?\]', raw, re.DOTALL)
                    selected_names = set(json.loads(arr.group())) if arr else set()
                    name_to_id = {v: k for k, v in candidates.items()}
                    next_frontier = [name_to_id[n] for n in selected_names if n in name_to_id]
                    if not next_frontier:
                        next_frontier = list(candidates.keys())[:max_expand]
                except Exception as exc:
                    print(f'[LLM_LocalRetriever] prune failed hop {hop}: {exc}')
                    next_frontier = list(candidates.keys())[:max_expand]

            visited.update(next_frontier)
            frontier = next_frontier
            if len(visited) >= self.max_nodes:
                break

        return self.graph.subgraph(list(visited)[:self.max_nodes]).copy()

    def retrieve(self, query: str) -> dict:
        eids = self._llm_query_entities(query)
        sg   = self._llm_guided_traverse(eids, query)
        return {
            'context':        self._to_context(sg),
            'retrieval_mode': 'llm_local',
            'entity_ids':     eids,
            'subgraph_nodes': sg.number_of_nodes(),
        }


# ── B. SemanticGlobalRetriever ────────────────────────────────────────────────
class SemanticGlobalRetriever(GlobalRetriever):
    """Cosine-similarity retrieval over community summaries.
    Requires sentence-transformers; falls back to keyword scoring if unavailable.
    """

    def __init__(self, communities: Dict[int, CommunityNode],
                 top_k: int = 3, embed_model: str = 'all-MiniLM-L6-v2'):
        super().__init__(communities, top_k)
        self._embed_model_name = embed_model
        self._encoder         = None
        self._cluster_embs    = None
        self._cluster_order: List[int] = []
        self._build_index()

    def _build_index(self):
        try:
            from sentence_transformers import SentenceTransformer
            self._encoder = SentenceTransformer(self._embed_model_name)
            summaries, self._cluster_order = [], []
            for cid, node in self.communities.items():
                text = node.summary or ' '.join(str(e) for e in node.entities[:10])
                summaries.append(text)
                self._cluster_order.append(cid)
            if summaries:
                self._cluster_embs = self._encoder.encode(summaries, convert_to_tensor=True)
            print(f'[SemanticGlobalRetriever] Indexed {len(summaries)} clusters.')
        except ImportError:
            print('[SemanticGlobalRetriever] sentence-transformers not installed — keyword fallback active.')

    def retrieve(self, query: str) -> dict:
        if self._encoder is None or self._cluster_embs is None:
            return super().retrieve(query)
        try:
            from sentence_transformers import util
            q_emb = self._encoder.encode(query, convert_to_tensor=True)
            cos   = util.cos_sim(q_emb, self._cluster_embs)[0]
            top_i = cos.argsort(descending=True)[:self.top_k].tolist()
            lines, clusters_used = ['[Global Semantic Context]'], []
            for idx in top_i:
                cid  = self._cluster_order[idx]
                node = self.communities[cid]
                if not node.summary:
                    continue
                score = float(cos[idx])
                lines.append(f'\n[{node.cluster_id}] L{node.level} (cos_sim={score:.3f}):')
                lines.append(f'  {node.summary}')
                lines.append(f'  Entities: {", ".join(str(e) for e in node.entities[:8])}')
                clusters_used.append(node.cluster_id)
            return {'context': '\n'.join(lines), 'retrieval_mode': 'semantic_global',
                    'clusters': clusters_used}
        except Exception as exc:
            print(f'[SemanticGlobalRetriever] {exc} — keyword fallback')
            return super().retrieve(query)


# ── C. AdvancedDualRetriever ──────────────────────────────────────────────────
class AdvancedDualRetriever(DualRetriever):
    """Drops in over DualRetriever.
    New mode 'llm_hybrid': llama3.2:3b merges + re-ranks local and global contexts.
    """

    _RERANK_PROMPT = (
        "You have two knowledge graph contexts for the query below.\n"
        "Synthesise them into a single concise context keeping only the most relevant facts.\n"
        "Do NOT add information not present in the contexts.\n\n"
        "Query: {query}\n\n"
        "Local Context:\n{local_ctx}\n\n"
        "Global Context:\n{global_ctx}\n\n"
        "Synthesised Context (max 400 words):"
    )

    def __init__(self, local: LLM_LocalRetriever,
                 global_: SemanticGlobalRetriever,
                 model: str = 'llama3.2:3b',
                 ollama_url: str = 'http://localhost:11434'):
        super().__init__(local, global_, model=model, ollama_url=ollama_url)
        self.local:   LLM_LocalRetriever    = local
        self.global_: SemanticGlobalRetriever = global_

    def _llm_call(self, prompt: str, timeout: int = 60) -> str:
        resp = requests.post(
            f'{self.ollama_url}/api/generate',
            json={'model': self.model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0}},
            timeout=timeout
        )
        resp.raise_for_status()
        return resp.json()['response']

    def retrieve(self, query: str, mode: str = 'auto') -> dict:
        if mode == 'llm_hybrid':
            l = self.local.retrieve(query)
            g = self.global_.retrieve(query)
            prompt = self._RERANK_PROMPT.format(
                query=query, local_ctx=l['context'], global_ctx=g['context'])
            try:
                merged = self._llm_call(prompt)
            except Exception as exc:
                merged = l['context'] + '\n\n' + g['context']
                print(f'[AdvancedDualRetriever] rerank failed: {exc}')
            return {'context': merged, 'retrieval_mode': 'llm_hybrid', 'local': l, 'global': g}
        return super().retrieve(query, mode)

    def answer(self, query: str, mode: str = 'auto') -> dict:
        ret = self.retrieve(query, mode)
        prompt = (
            'Answer the question using ONLY the provided knowledge graph context.\n'
            'If context is insufficient, say "Insufficient context."\n\n'
            f'Context:\n{ret["context"]}\n\nQuestion: {query}\n\nAnswer:'
        )
        try:
            answer = self._llm_call(prompt)
        except Exception as exc:
            answer = f'[Answer generation failed: {exc}]'
        return {'query': query, 'answer': answer,
                'context': ret['context'], 'retrieval_mode': ret['retrieval_mode']}


print('LLM_LocalRetriever, SemanticGlobalRetriever, AdvancedDualRetriever loaded.')


LLM_LocalRetriever, SemanticGlobalRetriever, AdvancedDualRetriever loaded.


## 10c. Conversational Retriever — Mistral-Nemo

Multi-turn conversation backed by knowledge graph retrieval.

**Flow per turn:**
1. **Query rewrite** — `mistral-nemo` condenses chat history + new question into a standalone retrieval query
2. **Graph retrieval** — `AdvancedDualRetriever` fetches context (`llm_hybrid` by default)
3. **Answer** — `mistral-nemo` generates a grounded response from context + history

Switch `conv_model='mistral-nemo'` to any model loaded via `ollama run <model>`.

In [51]:
class ConversationalRetriever:
    """Multi-turn conversational RAG using Mistral-Nemo with graph context.

    Usage:
        cr = ConversationalRetriever(adv_dual)
        cr.chat("What did Goldman Sachs acquire in 2022?")
        cr.chat("Who led that deal?")    # follow-up auto-uses history
        print(cr.history)
        cr.reset()
    """

    _REWRITE_PROMPT = (
        "Given the conversation history and a new user question, "
        "rewrite the question as a standalone search query capturing full context.\n"
        "Return ONLY the rewritten query — no explanation, no quotes.\n\n"
        "Conversation history:\n{history}\n\n"
        "New question: {question}\n\n"
        "Standalone query:"
    )

    _ANSWER_PROMPT = (
        "You are a financial knowledge assistant. "
        "Answer the user's question using the provided knowledge graph context and conversation history.\n"
        "Be concise and factual. If context is insufficient, say so explicitly.\n\n"
        "Knowledge Graph Context:\n{context}\n\n"
        "Conversation History:\n{history}\n\n"
        "User: {question}\n"
        "Assistant:"
    )

    def __init__(self, retriever: AdvancedDualRetriever,
                 conv_model: str = 'mistral-nemo',
                 retrieval_mode: str = 'llm_hybrid',
                 ollama_url: str = 'http://localhost:11434',
                 max_history_turns: int = 6):
        self.retriever       = retriever
        self.conv_model      = conv_model
        self.retrieval_mode  = retrieval_mode
        self.ollama_url      = ollama_url
        self.max_history_turns = max_history_turns
        self.history: List[Dict[str, str]] = []

    def _call(self, prompt: str, timeout: int = 90) -> str:
        resp = requests.post(
            f'{self.ollama_url}/api/generate',
            json={'model': self.conv_model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0.3, 'top_p': 0.9}},
            timeout=timeout
        )
        resp.raise_for_status()
        return resp.json()['response'].strip()

    def _format_history(self) -> str:
        if not self.history:
            return '(no prior turns)'
        lines = []
        for turn in self.history[-self.max_history_turns * 2:]:
            prefix = 'User' if turn['role'] == 'user' else 'Assistant'
            lines.append(f'{prefix}: {turn["content"]}')
        return '\n'.join(lines)

    def _rewrite_query(self, question: str) -> str:
        """Condense history + question into a standalone retrieval query via mistral-nemo."""
        if not self.history:
            return question
        prompt = self._REWRITE_PROMPT.format(
            history=self._format_history(), question=question)
        try:
            return self._call(prompt, timeout=30)
        except Exception as exc:
            print(f'[ConversationalRetriever] rewrite failed: {exc}')
            return question

    def chat(self, question: str, verbose: bool = True) -> str:
        """Send a message and return the assistant's response."""
        # 1. Rewrite for retrieval
        retrieval_query = self._rewrite_query(question)
        if verbose:
            print(f'  [retrieval query] {retrieval_query}')

        # 2. Retrieve graph context
        ret     = self.retriever.retrieve(retrieval_query, mode=self.retrieval_mode)
        context = ret['context']

        # 3. Generate answer with mistral-nemo
        prompt = self._ANSWER_PROMPT.format(
            context=context[:2500],
            history=self._format_history(),
            question=question,
        )
        try:
            answer = self._call(prompt)
        except Exception as exc:
            answer = f'[Generation failed: {exc}]'

        # 4. Update history
        self.history.append({'role': 'user',      'content': question})
        self.history.append({'role': 'assistant', 'content': answer})
        return answer

    def reset(self):
        """Clear conversation history."""
        self.history.clear()
        print('[ConversationalRetriever] History cleared.')

    def run_demo(self, questions: List[str]) -> None:
        """Run a canned multi-turn demo and print each exchange."""
        self.reset()
        print('=' * 65)
        print(f'ConversationalRetriever demo  model={self.conv_model}  retrieval={self.retrieval_mode}')
        print('=' * 65)
        for i, q in enumerate(questions, 1):
            print(f'\nTurn {i}  User: {q}')
            ans = self.chat(q, verbose=True)
            print(f'Assistant: {ans}')
        print('\n' + '=' * 65)


print('ConversationalRetriever (mistral-nemo) loaded.')


ConversationalRetriever (mistral-nemo) loaded.


In [52]:
# ── Advanced Retrieval + Conversational Demo ─────────────────────────────────
# Prerequisites:
#   ollama run llama3.2:3b     (entity extraction + reranking)
#   ollama run mistral-nemo    (conversational turns)

if OLLAMA_AVAILABLE:
    # Reuse the graph built in the main demo (gb + dual are in scope from cell 29/demo)
    _G   = dual.local.graph          # reuse existing graph
    _nel = NEL(ticker_anchors={})

    _llm_local  = LLM_LocalRetriever(_G, _nel, model='llama3.2:3b')
    _sem_global = SemanticGlobalRetriever(dual.global_.communities)
    adv_dual    = AdvancedDualRetriever(_llm_local, _sem_global, model='llama3.2:3b')

    # ── Retrieval mode comparison ─────────────────────────────────────────────
    TEST_Q = 'What companies did Goldman Sachs deal with?'
    print(f'\nRetrieval mode comparison — "{TEST_Q}"')
    print('-' * 55)
    for mode in ('local', 'llm_local', 'semantic_global', 'llm_hybrid'):
        r = adv_dual.retrieve(TEST_Q, mode=mode)
        ctx_len = len(r['context'])
        print(f'  [{mode:16s}] {ctx_len:4d} chars  ', end='')
        # Show first line of context
        first = r['context'].split('\n')[0]
        print(first[:60])

    # ── Conversational multi-turn with mistral-nemo ───────────────────────────
    print()
    conv = ConversationalRetriever(
        adv_dual,
        conv_model='mistral-nemo',
        retrieval_mode='llm_hybrid',
    )
    CONV_QUESTIONS = [
        'What major acquisitions did Goldman Sachs make?',
        'Who was the CEO during those deals?',
        'What was the strategic rationale behind the Marcus acquisition?',
    ]
    conv.run_demo(CONV_QUESTIONS)

else:
    print('[Skipped] Set OLLAMA_AVAILABLE=True and start llama3.2:3b + mistral-nemo via ollama')


[SemanticGlobalRetriever] sentence-transformers not installed — keyword fallback active.

Retrieval mode comparison — "What companies did Goldman Sachs deal with?"
-------------------------------------------------------
  [local           ]  194 chars  [Local KG Context]
  [llm_local       ]  194 chars  [Local KG Context]
  [semantic_global ]  194 chars  [Local KG Context]
  [llm_hybrid      ]  231 chars  Here is a concise context summarizing the most relevant fact

[ConversationalRetriever] History cleared.
ConversationalRetriever demo  model=mistral-nemo  retrieval=llm_hybrid

Turn 1  User: What major acquisitions did Goldman Sachs make?
  [retrieval query] What major acquisitions did Goldman Sachs make?
Assistant: [Generation failed: 404 Client Error: Not Found for url: http://localhost:11434/api/generate]

Turn 2  User: Who was the CEO during those deals?
[ConversationalRetriever] rewrite failed: 404 Client Error: Not Found for url: http://localhost:11434/api/generate
  [retrieval 

## 11. RAG Triad Evaluation (LLM-as-Judge)

Replaces F1-only evaluation with the full RAG quality triad:

| Metric | Definition |
|--------|-----------|
| **Faithfulness** | All answer claims grounded in the retrieved context |
| **Answer Relevance** | Answer addresses the question intent |
| **Context Precision** | Fraction of retrieved context relevant to the question |

LLM judge: Ollama (temp=0). Compatible with RAGAS / G-Eval frameworks.


In [53]:
class RAGTriadEvaluator:
    """Score RAG results with Faithfulness + Answer Relevance + Context Precision.
    Uses Ollama as judge (temperature=0). Each metric is 0.0–1.0.
    """

    _FAITH = (
        'Rate faithfulness of the answer to the context (0.0-1.0).\n'
        'Faithfulness = every claim in the answer is supported by the context.\n'
        '0.0 = hallucinated, 1.0 = fully grounded.\n\n'
        'Context: {context}\nAnswer: {answer}\n\n'
        'Output ONLY JSON: {{"score": <float>, "reason": "<one sentence>"}}'
    )
    _RELEV = (
        'Rate relevance of the answer to the question (0.0-1.0).\n'
        'Relevance = answer directly addresses the question.\n\n'
        'Question: {query}\nAnswer: {answer}\n\n'
        'Output ONLY JSON: {{"score": <float>, "reason": "<one sentence>"}}'
    )
    _PREC = (
        'Rate context precision (0.0-1.0).\n'
        'Precision = fraction of the context that is useful for answering the question.\n\n'
        'Question: {query}\nContext: {context}\n\n'
        'Output ONLY JSON: {{"score": <float>, "reason": "<one sentence>"}}'
    )

    def __init__(self, model: str = 'llama3.2:3b', dev_mode: bool = False,
                 ollama_url: str = 'http://localhost:11434'):
        self.model = 'llama3.2:1b' if dev_mode else model
        self.ollama_url = ollama_url

    def _judge(self, prompt: str) -> dict:
        try:
            resp = requests.post(f'{self.ollama_url}/api/generate',
                json={'model': self.model, 'prompt': prompt, 'stream': False,
                      'options': {'temperature': 0}},
                timeout=60)
            resp.raise_for_status()
            raw = resp.json()['response']
            hit = re.search(r'\{.*?\}', raw, re.DOTALL)
            if hit:
                return json.loads(hit.group())
        except Exception as exc:
            print(f'[RAGTriad judge] {exc}')
        return {'score': 0.0, 'reason': 'evaluation failed'}

    def score(self, result: dict) -> dict:
        query   = result.get('query', '')
        answer  = result.get('answer', '')
        context = result.get('context', '')[:2000]

        faith = self._judge(self._FAITH.format(context=context, answer=answer))
        relev = self._judge(self._RELEV.format(query=query, answer=answer))
        prec  = self._judge(self._PREC.format(query=query, context=context))

        f, r, p = float(faith['score']), float(relev['score']), float(prec['score'])
        return {
            'query':              query,
            'retrieval_mode':     result.get('retrieval_mode', '?'),
            'faithfulness':       round(f, 3),
            'answer_relevance':   round(r, 3),
            'context_precision':  round(p, 3),
            'rag_score':          round((f + r + p) / 3, 3),
            'faith_reason':       faith.get('reason', ''),
            'relev_reason':       relev.get('reason', ''),
            'prec_reason':        prec.get('reason', ''),
        }

    def evaluate_batch(self, results: List[dict]) -> List[dict]:
        out = []
        for r in results:
            print(f"  Scoring: {r.get('query','')[:60]}...")
            out.append(self.score(r))
        return out

    def summary(self, scores: List[dict]) -> dict:
        if not scores: return {}
        keys = ['faithfulness', 'answer_relevance', 'context_precision', 'rag_score']
        return {k: round(sum(s[k] for s in scores) / len(scores), 3) for k in keys}


# ── Baseline stubs (vector RAG comparison) ────────────────────────────────────
class VectorRAG_Baseline:
    """Naive cosine-similarity retrieval stub — no graph, pure embedding.
    Intended as ablation baseline. Requires sentence-transformers.
    """
    def answer(self, query: str, corpus: List[str]) -> dict:
        try:
            from sentence_transformers import SentenceTransformer, util
            _m = SentenceTransformer('all-MiniLM-L6-v2')
            q_emb = _m.encode(query, convert_to_tensor=True)
            c_emb = _m.encode(corpus, convert_to_tensor=True)
            scores = util.cos_sim(q_emb, c_emb)[0]
            top_idx = int(scores.argmax())
            context = corpus[top_idx]
        except ImportError:
            context = 'Vector baseline unavailable (sentence-transformers not installed).'
        return {'query': query, 'answer': context[:400],
                'context': context, 'retrieval_mode': 'vector_rag'}


print('RAGTriadEvaluator + VectorRAG_Baseline loaded.')


RAGTriadEvaluator + VectorRAG_Baseline loaded.


## 12. Pipeline Orchestrator + Ablation Engine

The `HierarchicalPipeline` wraps the full stack end-to-end.  
Call `build_rag()` after ingestion to get a ready `DualRetriever`.

Ablation dimensions (priority order):
1. `extraction_paradigm` — modular (snowball) vs joint_llm (ollama)
2. `anchoring` — ticker_on vs ticker_off
3. `retrieval_mode` — local vs global vs hybrid
4. `graph_scope` — global vs document_local


In [54]:
from dataclasses import dataclass as _dc


@_dc
class ExperimentConfig:
    exp_id:       str
    ner:          str
    nel:          str
    re:           str
    graph_scope:  str = 'global'
    ticker:       bool = True
    dev_mode:     bool = False


# ── Ablation configs (priority order) ─────────────────────────────────────────
ABLATION_CONFIGS = [
    # 1. Extraction paradigm
    {'name': 'modular_snowball',     'ner': 'spacy_trf', 'nel': 'fuzzy_flat', 're': 'snowball'},
    {'name': 'modular_openie',       'ner': 'spacy_trf', 'nel': 'fuzzy_flat', 're': 'openie'},
    {'name': 'joint_llm_ollama',     'ner': 'spacy_trf', 'nel': 'fuzzy_flat', 're': 'ollama', 'dev_mode': True},
    # 2. Ticker anchoring
    {'name': 'ticker_on',            'ner': 'spacy_trf', 'nel': 'fuzzy_flat', 're': 'snowball', 'ticker': True},
    {'name': 'ticker_off',           'ner': 'spacy_trf', 'nel': 'fuzzy_flat', 're': 'snowball', 'ticker': False},
    # 3. Graph scope
    {'name': 'scope_global',         'ner': 'spacy_trf', 'nel': 'fuzzy_flat', 're': 'snowball', 'graph_scope': 'global'},
    {'name': 'scope_document',       'ner': 'spacy_trf', 'nel': 'fuzzy_flat', 're': 'snowball', 'graph_scope': 'document_local'},
]


class HierarchicalPipeline:
    """End-to-end orchestrator.

    Flow:
      Preprocess → NER → NEL → Temporal → RE → Provenance → Graph
      → Community Detection → Community Summaries → DualRetriever
    """

    def __init__(self, chunk_size: int = 512, ticker_linking: bool = True,
                 ollama_url: str = 'http://localhost:11434',
                 dev_mode: bool = False):
        self._pp   = Preprocessor(chunk_size=chunk_size, ticker_linking=ticker_linking)
        self._tl   = TemporalLayer()
        self._prov = ProvenanceEnforcer()
        self.ollama_url = ollama_url
        self.dev_mode = dev_mode

    def _fallback_link(self, text: str) -> Link:
        return Link(entity=text,
                    canonical_id=f"LOCAL:{hashlib.md5(text.lower().encode()).hexdigest()[:8]}",
                    confidence=0.0, valid_time={'start': None, 'end': None})

    def process_document(self, doc_id: str, text: str, date: Optional[str] = None,
                         ner=None, nel=None, re_extractor=None, gb=None) -> List[dict]:
        chunks, ticker_anchors = self._pp.process(doc_id, text)
        if nel: nel.ticker_anchors.update(ticker_anchors)

        results = []
        for chunk in chunks:
            if ner: chunk.entities = ner.extract(chunk.text)
            if nel and chunk.entities:
                chunk.links = nel.link_all(chunk.entities, context=chunk.text)
            e2l = {l.entity: l for l in chunk.links}

            rels = re_extractor.extract(chunk.text, chunk.entities) if re_extractor and chunk.entities else []

            for r in rels:
                if not r.time.get('start'):
                    r.time = self._tl.infer_relation_time(r.provenance.raw_sentence or chunk.text, date)
                self._prov.enforce(r, chunk.chunk_id, chunk.text[:300])
            rels = self._prov.filter_valid(rels)
            chunk.relations = rels

            if gb:
                for r in rels:
                    ls = e2l.get(r.subj, self._fallback_link(r.subj))
                    lo = e2l.get(r.obj,  self._fallback_link(r.obj))
                    if not self._tl.is_valid(r.time, ls.valid_time) or \
                       not self._tl.is_valid(r.time, lo.valid_time):
                        continue
                    gb.add_relation(doc_id, ls, lo, r)
            results.append(chunk.to_dict())
        return results

    def build_rag(self, gb: GraphBuilder, communities: Optional[Dict] = None,
                  nel: Optional[NEL] = None, summarize: bool = True) -> DualRetriever:
        """Build DualRetriever from a populated GraphBuilder."""
        G = gb.get_graph()
        if communities is None:
            detector = CommunityDetector()
            level0 = detector.detect(G)
            communities = detector.build_hierarchy(G, level0)
            if summarize:
                summarizer = CommunitySummarizer(
                    dev_mode=self.dev_mode, ollama_url=self.ollama_url)
                communities = summarizer.summarize_all(communities)

        nel = nel or NEL(ticker_anchors={})
        local = LocalRetriever(G, nel)
        global_ = GlobalRetriever(communities)
        return DualRetriever(local, global_,
                             model='llama3.2:1b' if self.dev_mode else 'llama3.2:3b',
                             ollama_url=self.ollama_url)

    def run_experiment(self, cfg: ExperimentConfig, docs: List[dict]):
        try: ner = get_ner(cfg.ner)
        except Exception as exc:
            print(f'  [NER] {cfg.ner} failed ({exc}), fallback spacy_trf')
            ner = SpacyNER()

        nel = NEL(ticker_anchors={}, strategy=cfg.nel)

        try:
            re_kwargs = {'dev_mode': cfg.dev_mode, 'ollama_url': self.ollama_url} if cfg.re == 'ollama' else {}
            re_ext = get_re(cfg.re, **re_kwargs)
        except Exception as exc:
            print(f'  [RE] {cfg.re} failed ({exc}), fallback snowball')
            re_ext = Snowball_RE()

        gb = GraphBuilder(backend='networkx', scope=cfg.graph_scope)
        for doc in docs:
            self.process_document(
                doc_id=doc.get('doc_id', str(uuid.uuid4())),
                text=doc.get('text', ''), date=doc.get('date'),
                ner=ner, nel=nel, re_extractor=re_ext, gb=gb)
        return gb


class AblationEngine:
    """Config-driven ablation sweep.
    Stores results in results_dir/{exp_name}/metrics.json.
    """
    def __init__(self, configs=None, results_dir='./results'):
        self.configs = configs or ABLATION_CONFIGS
        self.results_dir = results_dir
        os.makedirs(results_dir, exist_ok=True)

    def run(self, pipeline: HierarchicalPipeline, docs: List[dict],
            eval_queries: Optional[List[str]] = None) -> List[dict]:
        evaluator = Evaluator(beta=0.5)
        rag_eval   = RAGTriadEvaluator(
            dev_mode=pipeline.dev_mode, ollama_url=pipeline.ollama_url)
        all_results = []

        for cfg_dict in self.configs:
            name = cfg_dict.get('name', str(uuid.uuid4())[:8])
            print(f'\n[Ablation] {name}')
            t0 = time.time()
            try:
                cfg = ExperimentConfig(
                    exp_id=name, ner=cfg_dict.get('ner','spacy_trf'),
                    nel=cfg_dict.get('nel','fuzzy_flat'), re=cfg_dict.get('re','snowball'),
                    graph_scope=cfg_dict.get('graph_scope','global'),
                    ticker=cfg_dict.get('ticker', True),
                    dev_mode=cfg_dict.get('dev_mode', False),
                )
                gb = pipeline.run_experiment(cfg, docs)
                gm = evaluator.evaluate_graph(gb)
                metrics = {'name': name, 'config': cfg_dict, 'graph': gm,
                           'elapsed': round(time.time()-t0,2), 'status': 'ok'}

                # RAG metrics (only if eval_queries provided and Ollama reachable)
                if eval_queries:
                    try:
                        dual = pipeline.build_rag(gb, summarize=True)
                        rag_results = [dual.answer(q) for q in eval_queries]
                        scores = rag_eval.evaluate_batch(rag_results)
                        metrics['rag'] = rag_eval.summary(scores)
                    except Exception as re_exc:
                        metrics['rag_error'] = str(re_exc)

                print(f"  nodes={gm['total_nodes']} edges={gm['total_edges']} "
                      f"prov={gm['provenance_pct']}% ts={gm['timestamp_pct']}%")
            except Exception as exc:
                metrics = {'name': name, 'config': cfg_dict,
                           'error': str(exc), 'elapsed': round(time.time()-t0,2), 'status': 'error'}
                print(f'  [Error] {exc}')

            exp_dir = os.path.join(self.results_dir, name)
            os.makedirs(exp_dir, exist_ok=True)
            with open(os.path.join(exp_dir, 'metrics.json'), 'w') as f:
                json.dump(metrics, f, indent=2)
            all_results.append(metrics)
        return all_results


# ── Legacy Evaluator (graph quality metrics) ──────────────────────────────────
Triple = Tuple[str, str, str]

def f_beta(p, r, beta=0.5):
    d = beta**2 * p + r
    return (1+beta**2)*p*r/d if d else 0.0

class Evaluator:
    def __init__(self, beta=0.5): self.beta = beta
    def evaluate_graph(self, gb: GraphBuilder) -> dict:
        G = gb.get_graph(); total = G.number_of_edges()
        invalid = sum(1 for _,_,d in G.edges(data=True)
                      if not d.get('provenance',{}).get('source_chunk_id')
                      or not d.get('provenance',{}).get('raw_sentence'))
        G_u = G.to_undirected()
        try: cc = nx.average_clustering(G_u) if G_u.number_of_nodes() else 0.0
        except: cc = 0.0
        s = gb.stats()
        return {'invalid_edge_rate': round(invalid/max(total,1),4),
                'clustering_coefficient': round(cc,4),
                'provenance_pct': s['provenance_pct'], 'timestamp_pct': s['timestamp_pct'],
                'total_nodes': s['nodes'], 'total_edges': total,
                'conflict_groups': s['conflict_groups']}


print('HierarchicalPipeline + AblationEngine + Evaluator loaded.')


HierarchicalPipeline + AblationEngine + Evaluator loaded.


## 13. Demo — Hierarchical GraphRAG on Sample Bloomberg Docs

Full pipeline run: extraction → graph → Leiden communities → LLM summaries → dual retrieval → RAG triad scoring.

> **Dev mode** (`dev_mode=True`): uses `llama3.2:1b`, skips Ollama calls if unavailable.  
> Set `OLLAMA_AVAILABLE = True` once `ollama run llama3.2:3b` is running.


In [55]:
OLLAMA_AVAILABLE = True   # set True when: ollama run llama3.2:3b
DEV_MODE         = True    # llama3.2:1b for fast dev iteration

SAMPLE_DOCS = [
    {'doc_id': 'bloomberg_001',
     'text': 'Goldman Sachs (GS:US) acquired Marcus Investments in 2022, expanding its '
             'retail banking unit. CEO David Solomon said the deal was valued at $2.2B.',
     'date': '2022-03-15'},
    {'doc_id': 'bloomberg_002',
     'text': 'Apple (AAPL:US) and Microsoft (MSFT:US) announced a joint partnership in '
             'Q1 2023 to develop AI infrastructure. The deal follows Microsoft\'s $10B '
             'investment in OpenAI.',
     'date': '2023-02-01'},
    {'doc_id': 'bloomberg_003',
     'text': 'JP Morgan Chase reported record profits in early 2024 after acquiring '
             'First Republic Bank. The acquisition was valued at $10.6B.',
     'date': '2024-01-10'},
]

EVAL_QUERIES = [
    'What did Goldman Sachs acquire in 2022?',          # → local (entity present)
    'Which companies formed partnerships in 2023?',     # → global (thematic)
    'What acquisitions happened in the banking sector?', # → global
    'What is the relationship between GS:US and Marcus?',# → local
]

print('=' * 65)
print('Hierarchical GraphRAG — Demo')
print('=' * 65)

# ── Step 1: Ingest + Graph Build ──────────────────────────────────────────────
pipeline = HierarchicalPipeline(chunk_size=256, ticker_linking=True,
                                 dev_mode=DEV_MODE)
ner    = SpacyNER()
nel    = NEL(ticker_anchors={}, strategy='fuzzy_flat')
re_ext = Snowball_RE()
gb     = GraphBuilder(backend='networkx', scope='global')

print('\n[1] Extracting relations...')
for doc in SAMPLE_DOCS:
    res = pipeline.process_document(
        doc_id=doc['doc_id'], text=doc['text'], date=doc['date'],
        ner=ner, nel=nel, re_extractor=re_ext, gb=gb)
    for r in res:
        rels = [(x['subj'], x['rel'], x['obj']) for x in r['relations']]
        if rels: print(f"  {doc['doc_id']}: {rels}")

print('\n[2] Graph stats:')
for k, v in gb.stats().items():
    print(f'  {k}: {v}')

# ── Step 2: Community Detection ───────────────────────────────────────────────
print('\n[3] Community detection (Leiden / networkx fallback)...')
detector = CommunityDetector()
G = gb.get_graph()
level0 = detector.detect(G)
communities = detector.build_hierarchy(G, level0)
print(f'  Node→cluster mapping: {level0}')

# ── Step 3: Community Summaries (Ollama or stub) ──────────────────────────────
print('\n[4] Community summarization...')
if OLLAMA_AVAILABLE:
    summarizer = CommunitySummarizer(dev_mode=DEV_MODE)
    communities = summarizer.summarize_all(communities)
else:
    print('  [Ollama unavailable] — inserting stub summaries.')
    for cid, node in communities.items():
        entity_names = ', '.join(str(e) for e in node.entities[:5])
        node.summary = (f'Cluster {node.cluster_id} (L{node.level}) contains entities: '
                        f'{entity_names}. Key relations: '
                        f'{len(node.key_relations)} edges within cluster.')

for cid, node in communities.items():
    print(f'  [{node.cluster_id}] {node.summary[:100]}...')

# ── Step 4: Build DualRetriever ───────────────────────────────────────────────
print('\n[5] Building DualRetriever...')
local_retr  = LocalRetriever(G, nel)
global_retr = GlobalRetriever(communities)
dual        = DualRetriever(local_retr, global_retr, model='llama3.2:1b' if DEV_MODE else 'llama3.2:3b')

# ── Step 5: Retrieval (context only, no Ollama answer) ────────────────────────
print('\n[6] Retrieval demo (context only):')
for q in EVAL_QUERIES:
    ret = dual.retrieve(q, mode='auto')
    print(f'\n  Q: {q}')
    print(f'  mode={ret["retrieval_mode"]}')
    print(f'  context[:120]: {ret["context"][:120].strip()}')

# ── Step 6: Full answer + RAG scoring (requires Ollama) ─────────────────────
print('\n[7] Full RAG answers + triad scoring:')
if OLLAMA_AVAILABLE:
    rag_results = [dual.answer(q) for q in EVAL_QUERIES]
    rag_eval = RAGTriadEvaluator(dev_mode=DEV_MODE)
    scores = rag_eval.evaluate_batch(rag_results)
    summary = rag_eval.summary(scores)
    print('\n  RAG Triad Summary:')
    for k, v in summary.items():
        bar = '#' * int(v * 20)
        print(f'    {k:22s} {v:.3f}  {bar}')
    print('\n  Per-query scores:')
    for s in scores:
        print(f'    [{s["retrieval_mode"]:7s}] F={s["faithfulness"]:.2f} '
              f'R={s["answer_relevance"]:.2f} P={s["context_precision"]:.2f} '
              f'→ RAG={s["rag_score"]:.2f}  | {s["query"][:45]}')
else:
    print('  [Ollama unavailable] — set OLLAMA_AVAILABLE=True to run full RAG eval.')

print('\n' + '=' * 65)
print('Demo complete.')
print('=' * 65)


Hierarchical GraphRAG — Demo

[1] Extracting relations...
  bloomberg_001: [('Goldman Sachs', 'acquired', 'Marcus Investments')]

[2] Graph stats:
  nodes: 2
  edges: 1
  provenance_pct: 100.0
  timestamp_pct: 100.0
  conflict_groups: 0

[3] Community detection (Leiden / networkx fallback)...
Hierarchy built: 1 L0 clusters, 1 L1 super-clusters.
  Node→cluster mapping: {'LEI:549300IUPX6KIT9GK103': 0, 'LOCAL:9daa70b3': 0}

[4] Community summarization...
Summarizing 2 community nodes...
  L0_C0 (L0, 2 entities)
[CommunitySummarizer] L0_C0: 404 Client Error: Not Found for url: http://localhost:11434/api/generate
  L1_SC0 (L1, 2 entities)
[CommunitySummarizer] L1_SC0: 404 Client Error: Not Found for url: http://localhost:11434/api/generate
  [L0_C0] [Summarization failed for L0_C0]...
  [L1_SC0] [Summarization failed for L1_SC0]...

[5] Building DualRetriever...

[6] Retrieval demo (context only):

  Q: What did Goldman Sachs acquire in 2022?
  mode=local
  context[:120]: [Local KG Context]

## 14. Continuous Testing

In [56]:
def test_provenance_invariant(results: List[dict]) -> List[str]:
    errors = []
    for r in results:
        for rel in r.get('relations', []):
            prov = rel.get('provenance', {})
            if not prov.get('source_chunk_id'):
                errors.append(f'FAIL missing source_chunk_id: {rel["subj"]}→{rel["obj"]}')
            if not prov.get('raw_sentence'):
                errors.append(f'FAIL missing raw_sentence: {rel["subj"]}→{rel["obj"]}')
        for lnk in r.get('links', []):
            if not lnk.get('canonical_id'):
                errors.append(f'FAIL missing canonical_id: {lnk}')
    return errors


def test_append_only() -> Tuple[bool, str]:
    _gb = GraphBuilder()
    ls = Link('GS', 'LEI:GS001', 0.9, {'start': '2020-01-01', 'end': None})
    lo = Link('Marcus', 'LOCAL:marc', 0.5, {'start': None, 'end': None})
    r1 = Relation('GS', 'acquired', 'Marcus', {'start': '2022-01-01', 'end': None},
                  Provenance('c1', 'GS acquired Marcus in 2022.'), 0.8)
    r2 = Relation('GS', 'merger_cancelled', 'Marcus', {'start': '2023-01-01', 'end': None},
                  Provenance('c2', 'GS merger cancelled.'), 0.7)
    _gb.add_relation('doc', ls, lo, r1)
    _gb.add_relation('doc', ls, lo, r2)
    n = _gb.get_graph().number_of_edges()
    return n == 2, f'expected 2 edges, got {n}'


def test_community_schema(communities: Dict) -> List[str]:
    errors = []
    for cid, node in communities.items():
        if not isinstance(node.cluster_id, str) or not node.cluster_id:
            errors.append(f'FAIL missing cluster_id for key {cid}')
        if node.level not in (0, 1, 2):
            errors.append(f'FAIL invalid level {node.level} for {node.cluster_id}')
        if not isinstance(node.entities, list):
            errors.append(f'FAIL entities not a list for {node.cluster_id}')
    return errors


def test_router(nel: NEL) -> List[str]:
    errors = []
    _local = LocalRetriever(nx.MultiDiGraph(), nel)
    _global = GlobalRetriever({})
    _dual = DualRetriever(_local, _global)
    entity_q  = 'What did Goldman Sachs acquire?'
    thematic_q = 'What sectors saw consolidation?'
    if not _dual._has_entities(entity_q):
        errors.append(f'FAIL router should detect entities in: {entity_q}')
    if _dual._has_entities(thematic_q):
        errors.append(f'FAIL router should not detect entities in: {thematic_q}')
    return errors


print('=' * 55)
print('Invariant Tests')
print('=' * 55)

# Provenance
all_results_flat = []
for doc in SAMPLE_DOCS:
    _pp2 = HierarchicalPipeline(chunk_size=256)
    _nel2 = NEL(ticker_anchors={}, strategy='fuzzy_flat')
    _gb2 = GraphBuilder()
    all_results_flat.extend(_pp2.process_document(
        doc_id=doc['doc_id'], text=doc['text'], date=doc['date'],
        ner=SpacyNER(), nel=_nel2, re_extractor=Snowball_RE(), gb=_gb2))

prov_errs = test_provenance_invariant(all_results_flat)
print(f'[{"PASS" if not prov_errs else "FAIL"}] Provenance invariant'
      + (f' — {prov_errs[:2]}' if prov_errs else ''))

ao_ok, ao_msg = test_append_only()
print(f'[{"PASS" if ao_ok else "FAIL"}] Append-only — {ao_msg}')

comm_errs = test_community_schema(communities)
print(f'[{"PASS" if not comm_errs else "FAIL"}] Community schema'
      + (f' — {comm_errs[:2]}' if comm_errs else ''))

router_errs = test_router(nel)
print(f'[{"PASS" if not router_errs else "FAIL"}] Retrieval router'
      + (f' — {router_errs}' if router_errs else ''))


Invariant Tests
[PASS] Provenance invariant
[PASS] Append-only — expected 2 edges, got 2
[PASS] Community schema
[PASS] Retrieval router


## 15. Ablation Sweep

> Set `RUN_ABLATION = True` to execute.  
> Set `OLLAMA_AVAILABLE = True` to include RAG triad scoring in ablations.

Priority order: `extraction_paradigm` → `anchoring` → `retrieval_mode` → `graph_scope`


In [57]:
RUN_ABLATION = False

QUICK_ABLATION = [
    {'name': 'modular_snowball_global',  'ner': 'spacy_trf', 'nel': 'fuzzy_flat', 're': 'snowball', 'graph_scope': 'global'},
    {'name': 'modular_snowball_doc',     'ner': 'spacy_trf', 'nel': 'fuzzy_flat', 're': 'snowball', 'graph_scope': 'document_local'},
    {'name': 'ticker_on',                'ner': 'spacy_trf', 'nel': 'fuzzy_flat', 're': 'snowball', 'ticker': True},
    {'name': 'ticker_off',               'ner': 'spacy_trf', 'nel': 'fuzzy_flat', 're': 'snowball', 'ticker': False},
    {'name': 'joint_llm_ollama_dev',     'ner': 'spacy_trf', 'nel': 'fuzzy_flat', 're': 'ollama',   'dev_mode': True},
]

if RUN_ABLATION:
    abl_pipeline = HierarchicalPipeline(dev_mode=DEV_MODE)
    ablation     = AblationEngine(configs=QUICK_ABLATION, results_dir='./results')
    eval_q       = EVAL_QUERIES if OLLAMA_AVAILABLE else None
    ab_results   = ablation.run(abl_pipeline, SAMPLE_DOCS, eval_queries=eval_q)

    print('\n' + '=' * 75)
    print(f'  {"Experiment":<35} {"Status":<6} {"Nodes":>5} {"Edges":>5} {"Prov%":>6} {"RAG":>6}')
    print('=' * 75)
    for m in ab_results:
        gm  = m.get('graph', {})
        rag = m.get('rag', {})
        print(f'  {m["name"]:<35} {m["status"]:<6} '
              f'{gm.get("total_nodes","-"):>5} {gm.get("total_edges","-"):>5} '
              f'{str(gm.get("provenance_pct","-")):>5}% '
              f'{str(rag.get("rag_score","n/a")):>6}')
else:
    print('Ablation skipped. Set RUN_ABLATION = True to execute.')


Ablation skipped. Set RUN_ABLATION = True to execute.


## End-to-End Flow

```
Raw Bloomberg Text
  ↓  Preprocessing       (ticker anchoring, normalize, chunk)
  ↓  NER                 (spacy_trf | finbert_ner | gliner)
  ↓  NEL                 (ticker → alias → fuzzy → LOCAL fingerprint)
  ↓  Temporal            (date extraction, PIT validity enforcement)
  ↓  RE ──────────────── [ABLATION: modular vs joint_llm]
  │     Modular:  Snowball | OpenIE
  │     Joint LLM: Ollama / Llama 3.2:3b (temp=0, strict schema)
  ↓  Provenance          (source_chunk_id + raw_sentence — hard invariant)
  ↓  Graph Builder       (append-only, versioned, NetworkX / Neo4j)
  ↓  Community Detection (Leiden → 3-level hierarchy: L0 / L1 / L2)
  ↓  Cluster Summaries   (Ollama, grounded in graph edges only)
  ↓  Dual Retrieval ──── [ABLATION: local vs global vs hybrid]
  │     Local:   entity → graph traversal → subgraph context
  │     Global:  keyword → cluster summary → hierarchical context
  ↓  Answer Generation   (Ollama, context-grounded, temp=0)
  ↓  RAG Triad Eval      (Faithfulness + Answer Relevance + Context Precision)
```

**Hard invariants**
- No relation without `provenance` (source_chunk_id + raw_sentence)
- No entity without `canonical_id`
- No relation without `time` (or explicit `null`)
- Append-only graph (parallel edges, no overwrite)
- All summaries derived from graph edges only — no external knowledge
- Deterministic LLM calls (temperature = 0)

**Ablation axes**
1. `extraction_paradigm` — modular (Snowball/OpenIE) vs joint_llm (Ollama)
2. `anchoring`           — ticker_on vs ticker_off
3. `retrieval_mode`      — local vs global vs hybrid
4. `graph_scope`         — global vs document_local

**Failure surfaces**
1. NEL ambiguity → hierarchy errors (LOCAL fingerprint fallback)
2. Temporal inconsistency → PIT validity gate drops relation
3. LLM extraction drift → schema validation rejects non-whitelisted rel types
4. Community summarization hallucination → grounding constraint (edges-only prompt)
